In [1]:
# Imports and Configuration

import os
import numpy as np
import pickle
import json
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Paths
FEATURES_DIR = 'processed_features'
OUTPUT_DIR = 'models_urdu_sl'

# Model settings
LSTM_UNITS = 64
DROPOUT = 0.6
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

print("MODEL TRAINING")
print(f"Features directory: {FEATURES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL TRAINING
Features directory: processed_features
Output directory: models_urdu_sl


In [2]:
# Load Preprocessed Features
print("\nLoading preprocessed features...")

# Load features
X_train = np.load(os.path.join(FEATURES_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(FEATURES_DIR, 'y_train.npy'))
X_val = np.load(os.path.join(FEATURES_DIR, 'X_val.npy'))
y_val = np.load(os.path.join(FEATURES_DIR, 'y_val.npy'))
X_test = np.load(os.path.join(FEATURES_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(FEATURES_DIR, 'y_test.npy'))

print(f"Features loaded:")
print(f"   Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"   Val:   {X_val.shape}, Labels: {y_val.shape}")
print(f"   Test:  {X_test.shape}, Labels: {y_test.shape}")

# Load label encoder
label_encoder_path = os.path.join(FEATURES_DIR, 'label_encoder.pkl')
with open(label_encoder_path, 'rb') as f:
    label_encoder = pickle.load(f)
print(f"\nLabel encoder loaded")
print(f"   Classes: {label_encoder.classes_}")

# Load metadata
metadata_path = os.path.join(FEATURES_DIR, 'metadata.json')
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
print(f"\nMetadata loaded:")
print(f"   Sequence length: {metadata['sequence_length']}")
print(f"   Feature dimension: {metadata['feature_dim']}")
print(f"   Number of classes: {metadata['num_classes']}")
print(f"   Augmentation factor: {metadata['augmentation_factor']:.1f}x")


Loading preprocessed features...
Features loaded:
   Train: (1872, 30, 258), Labels: (1872,)
   Val:   (101, 30, 258), Labels: (101,)
   Test:  (101, 30, 258), Labels: (101,)

Label encoder loaded
   Classes: ['آپ بہت خوبصورت ہیں۔' 'آپ کا نام کیا ہے؟' 'آہِستہ' 'ادھر آو!'
 'السلام وعلیکم' 'آپ کہاں رہتے ہیں؟' 'آپ کیسے ہیں؟'
 'ایمبولینس کو کال کریں' 'بھائی' 'حالات کیسے ہیں؟' 'خوش رہو' 'شکریہ'
 'فون کہاں ہے؟' 'مبارک ہو' 'میں اشاروں کی زبان سیکھ رہا ہوں'
 'میں واش روم جا سکتا ہوں؟' 'میں ٹھیک ہوں' 'نہیں' 'چھونا مت' 'کمپیوٹر'
 'کیا آپ تیار ہیں؟' 'کیا آپ پاس یا ناکام ہوئے ہیں؟' 'کیا تم بہرے ہو؟'
 'کیا میں اپ کی مدد کر سکتا ہوں؟' 'گھبرائیں نہیں']

Metadata loaded:
   Sequence length: 30
   Feature dimension: 258
   Number of classes: 25
   Augmentation factor: 4.0x


In [3]:
# Build Model
print("\nBuilding model...")

SEQUENCE_LENGTH = metadata['sequence_length']
FEATURE_DIM = metadata['feature_dim']
num_classes = metadata['num_classes']

model = Sequential([
    # Regular LSTM
    LSTM(LSTM_UNITS, return_sequences=False, input_shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
    Dropout(DROPOUT),
    
    # Dense layers
    Dense(64, activation='relu'),
    Dropout(DROPOUT),
    
    # Output layer
    Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model built successfully")
model.summary()


Building model...
Model built successfully


C:\Users\mypc4\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          82,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 25)                  │           1,625 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 88,473 (345.60 KB)

 Trainable params: 88,473 (345.60 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Train Model
print("\nTraining model...")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = f'urdu_sl_model_{timestamp}'

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, f'{model_name}_best.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


Training model...
Epoch 1/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0506 - loss: 3.2467
Epoch 1: val_accuracy improved from -inf to 0.11881, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.0513 - loss: 3.2438 - val_accuracy: 0.1188 - val_loss: 2.9990 - learning_rate: 0.0010
Epoch 2/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1173 - loss: 3.0176
Epoch 2: val_accuracy improved from 0.11881 to 0.25743, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1175 - loss: 3.0149 - val_accuracy: 0.2574 - val_loss: 2.5139 - learning_rate: 0.0010
Epoch 3/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1618 - loss: 2.6961
Epoch 3: val_accuracy improved from 0.25743 to 0.51485, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.1620 - loss: 2.6947 - val_accuracy: 0.5149 - val_loss: 2.0158 - learning_rate: 0.0010
Epoch 4/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2236 - loss: 2.3624
Epoch 4: val_accuracy improved from 0.51485 to 0.62376, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.2243 - loss: 2.3583 - val_accuracy: 0.6238 - val_loss: 1.6299 - learning_rate: 0.0010
Epoch 5/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3066 - loss: 2.0610
Epoch 5: val_accuracy improved from 0.62376 to 0.73267, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


...

59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5139 - loss: 1.3262 - val_accuracy: 0.8713 - val_loss: 0.5811 - learning_rate: 0.0010
Epoch 10/50
54/59 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5732 - loss: 1.1506
Epoch 10: val_accuracy improved from 0.87129 to 0.97030, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5741 - loss: 1.1492 - val_accuracy: 0.9703 - val_loss: 0.4729 - learning_rate: 0.0010
Epoch 11/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6254 - loss: 1.0925
Epoch 11: val_accuracy did not improve from 0.97030
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6257 - loss: 1.0914 - val_accuracy: 0.9406 - val_loss: 0.3823 - learning_rate: 0.0010
Epoch 12/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7044 - loss: 0.8744
Epoch 12: val_accuracy did not improve from 0.97030
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7048 - loss: 0.8729 - val_accuracy: 0.9703 - val_loss: 0.2930 - learning_rate: 0.0010
Epoch 13/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7009 - loss: 0.8458
Epoch 13: val_accuracy improved from 0.97030 to 0.98020, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7009 - loss: 0.8459 - val_accuracy: 0.9802 - val_loss: 0.2688 - learning_rate: 0.0010
Epoch 14/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6972 - loss: 0.8384
Epoch 14: val_accuracy did not improve from 0.98020
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6979 - loss: 0.8370 - val_accuracy: 0.9604 - val_loss: 0.2364 - learning_rate: 0.0010
Epoch 15/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7998 - loss: 0.6397
Epoch 15: val_accuracy did not improve from 0.98020
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7996 - loss: 0.6392 - val_accuracy: 0.9406 - val_loss: 0.2785 - learning_rate: 0.0010
Epoch 16/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7674 - loss: 0.6873
Epoch 16: val_accuracy did not improve from 0.98020
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7675 - loss: 0.6875 - val_accuracy: 0.9604 - val_loss: 0.1979 - learning_rate: 0.0010
Epoch 17/50
57/59 ━━━━━━━━

59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7572 - loss: 0.7739 - val_accuracy: 0.9901 - val_loss: 0.1554 - learning_rate: 0.0010
Epoch 25/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8259 - loss: 0.5168
Epoch 25: val_accuracy did not improve from 0.99010
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8262 - loss: 0.5164 - val_accuracy: 0.9901 - val_loss: 0.0736 - learning_rate: 0.0010
Epoch 26/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8778 - loss: 0.4018
Epoch 26: val_accuracy did not improve from 0.99010
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8781 - loss: 0.4010 - val_accuracy: 0.9901 - val_loss: 0.0848 - learning_rate: 0.0010
Epoch 27/50
56/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8615 - loss: 0.3962
Epoch 27: val_accuracy improved from 0.99010 to 1.00000, saving model to models_urdu_sl\urdu_sl_model_20251027_025844_best.h5


59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8619 - loss: 0.3958 - val_accuracy: 1.0000 - val_loss: 0.0310 - learning_rate: 0.0010
Epoch 28/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8720 - loss: 0.3701
Epoch 28: val_accuracy did not improve from 1.00000
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8724 - loss: 0.3690 - val_accuracy: 1.0000 - val_loss: 0.0269 - learning_rate: 0.0010
Epoch 29/50
57/59 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8886 - loss: 0.3257
Epoch 29: val_accuracy did not improve from 1.00000
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8880 - loss: 0.3269 - val_accuracy: 0.9802 - val_loss: 0.0371 - learning_rate: 0.0010
Epoch 30/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8877 - loss: 0.3289
Epoch 30: val_accuracy did not improve from 1.00000
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8878 - loss: 0.3291 - val_accuracy: 0.9802 - val_loss: 0.1070 - learning_rate: 0.0010
Epoch 31/50
56/59 ━━━━━━━━

In [5]:
# Evaluate Model
print("\nEvaluating model...")

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
best_val_accuracy = max(history.history['val_accuracy'])
best_train_accuracy = max(history.history['accuracy'])

print(f"\nRESULTS:")
print(f"   Train Accuracy:      {best_train_accuracy*100:.2f}%")
print(f"   Validation Accuracy: {best_val_accuracy*100:.2f}%")
print(f"   Test Accuracy:       {test_accuracy*100:.2f}%")

gap = best_val_accuracy - test_accuracy
print(f"\nOverfitting Check:")
print(f"   Val-Test Gap: {gap*100:.2f}%")

if gap < 0.05:
    print(f"   Model generalizes well")
elif gap < 0.10:
    print(f"   Acceptable generalization")
else:
    print(f"   Some overfitting detected")


Evaluating model...

RESULTS:
   Train Accuracy:      91.51%
   Validation Accuracy: 100.00%
   Test Accuracy:       100.00%

Overfitting Check:
   Val-Test Gap: 0.00%
   Model generalizes well


In [6]:
# Per-Class Accuracy
print(f"\nPer-gesture accuracy:")

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

for class_idx, gesture in enumerate(label_encoder.classes_):
    mask = y_test == class_idx
    if np.sum(mask) > 0:
        class_acc = np.mean(y_pred_classes[mask] == class_idx)
        print(f"   {gesture}: {class_acc*100:.1f}%")


Per-gesture accuracy:
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
   آپ بہت خوبصورت ہیں۔: 100.0%
   آپ کا نام کیا ہے؟: 100.0%
   آہِستہ: 100.0%
   ادھر آو!: 100.0%
   السلام وعلیکم: 100.0%
   آپ کہاں رہتے ہیں؟: 100.0%
   آپ کیسے ہیں؟: 100.0%
   ایمبولینس کو کال کریں: 100.0%
   بھائی: 100.0%
   حالات کیسے ہیں؟: 100.0%
   خوش رہو: 100.0%
   شکریہ: 100.0%
   فون کہاں ہے؟: 100.0%
   مبارک ہو: 100.0%
   میں اشاروں کی زبان سیکھ رہا ہوں: 100.0%
   میں واش روم جا سکتا ہوں؟: 100.0%
   میں ٹھیک ہوں: 100.0%
   نہیں: 100.0%
   چھونا مت: 100.0%
   کمپیوٹر: 100.0%
   کیا آپ تیار ہیں؟: 100.0%
   کیا آپ پاس یا ناکام ہوئے ہیں؟: 100.0%
   کیا تم بہرے ہو؟: 100.0%
   کیا میں اپ کی مدد کر سکتا ہوں؟: 100.0%
   گھبرائیں نہیں: 100.0%


In [7]:
# Save Model and Results
print("\nSaving model...")

# Save model
model_path = os.path.join(OUTPUT_DIR, f'{model_name}.h5')
model.save(model_path)
print(f"Model saved: {model_path}")

# Save label encoder
label_encoder_path = os.path.join(OUTPUT_DIR, f'{model_name}_labels.pkl')
with open(label_encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder saved: {label_encoder_path}")

# Save training metadata
training_metadata = {
    'timestamp': timestamp,
    'model_name': model_name,
    'train_samples': int(len(X_train)),
    'val_samples': int(len(X_val)),
    'test_samples': int(len(X_test)),
    'num_classes': num_classes,
    'sequence_length': SEQUENCE_LENGTH,
    'feature_dim': FEATURE_DIM,
    'lstm_units': LSTM_UNITS,
    'dropout': DROPOUT,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epochs_trained': len(history.history['loss']),
    'best_train_accuracy': float(best_train_accuracy),
    'best_val_accuracy': float(best_val_accuracy),
    'test_accuracy': float(test_accuracy),
    'val_test_gap': float(gap),
    'class_names': label_encoder.classes_.tolist()
}

training_metadata_path = os.path.join(OUTPUT_DIR, f'{model_name}_metadata.json')
with open(training_metadata_path, 'w', encoding='utf-8') as f:
    json.dump(training_metadata, f, indent=4, ensure_ascii=False)
print(f"Training metadata saved: {training_metadata_path}")

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Model architecture: Regular LSTM")
print(f"Training samples: {len(X_train)} (augmented)")
print(f"Validation samples: {len(X_val)} (original)")
print(f"Test samples: {len(X_test)} (original)")
print(f"Test accuracy: {test_accuracy*100:.2f}%")
print("="*70)


Saving model...
Model saved: models_urdu_sl\urdu_sl_model_20251027_025844.h5
Label encoder saved: models_urdu_sl\urdu_sl_model_20251027_025844_labels.pkl
Training metadata saved: models_urdu_sl\urdu_sl_model_20251027_025844_metadata.json

TRAINING COMPLETE
Model architecture: Regular LSTM
Training samples: 1872 (augmented)
Validation samples: 101 (original)
Test samples: 101 (original)
Test accuracy: 100.00%
